## Imports

In [51]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split

## Loading and Viewing the Datasets

In [5]:
# creating a variable to store the absolute .csv file path for the full Deep4Chem dataset
d4c_path = r'C:/Users/Amy/Desktop/Masters_Project/paper_methods/Deep4Chem_chemprop/Deep4Chem_chemprop/d4c_ext_coef_all_data.csv'

# reading in the dataset as stored in the path variable
d4c = pd.read_csv(d4c_path)

# printing the index and heading of each column in the dataset
for i, col in enumerate(d4c.columns):
    print(i, repr(col))

# viewing the shape (no. entries) and title of columns contained by the data
print('Shape:', d4c.shape)
print('\nColumns:')
for col in d4c.columns:
    print(repr(col))

# checking for missing values within the dataset
print('\nMissing values:')
print(d4c.isnull().sum())

0 'Chromophore'
1 'Solvent'
2 'Absorption max (nm)'
3 'log(e/mol-1 dm3 cm-1)'
Shape: (8032, 4)

Columns:
'Chromophore'
'Solvent'
'Absorption max (nm)'
'log(e/mol-1 dm3 cm-1)'

Missing values:
Chromophore              0
Solvent                  0
Absorption max (nm)      0
log(e/mol-1 dm3 cm-1)    0
dtype: int64


In [72]:
# creating a variable to store the absolute .csv file path for the full Reaxys dataset ("no_numbers" = no Reaxys registry numbers as final column)
reaxys_path = r'C:/Users/Amy/Desktop/Masters_Project/paper_methods/reaxys_dataset/reaxys_data_no_numbers.csv'

# reading in the dataset as stored in the path variable
reaxys = pd.read_csv(reaxys_path)

# printing the index and heading of each column in the dataset
for i, col in enumerate(reaxys.columns):
    print(i, repr(col))

# viewing the shape (no. entries) and title of columns contained by the data
print('Shape:', reaxys.shape)
print('\nColumns:')
for col in reaxys.columns:
    print(repr(col))

# checking for missing values within the dataset
print('\nMissing values:')
print(reaxys.isnull().sum())

0 'Chromophore'
1 'Solvent'
2 'Absorption max (nm)'
3 'log(e/mol-1 dm3 cm-1)'
Shape: (55866, 4)

Columns:
'Chromophore'
'Solvent'
'Absorption max (nm)'
'log(e/mol-1 dm3 cm-1)'

Missing values:
Chromophore              0
Solvent                  0
Absorption max (nm)      0
log(e/mol-1 dm3 cm-1)    0
dtype: int64


### Comments:
8,032 datapoints in Deep4Chem dataset representing ~3.8k unique chromophores, 55,866 in Reaxys representing ~38k chromophores as reported

No missing values in either => handy for handling datasets later

Column names were initially slightly different and in different orders in Reaxys - fixed so following scripts will be generalisable

## Train/Test Split

In [7]:
# retrieving number of unique molecules (chromophores) in the Deep4Chem dataset
unique_mols = d4c['Chromophore'].unique()
print(len(unique_mols))

# creating training and testing subsets of specified sizes w.r.t chromophore SMILES entry - dataframes created include molecules specified in the 
# following line by the 'unique_mols' dataframe, resulting in the subsets being of lengths determined by the number of unique molecule + solvent 
# combinations that correspond
train_mols, test_mols = train_test_split(unique_mols, test_size=1000, random_state=0)
train_df = d4c[d4c['Chromophore'].isin(train_mols)]
test_df = d4c[d4c['Chromophore'].isin(test_mols)]

# printing the shape of each subset, as well as a confirmation that the test set contains 1000 molecules while the rest comprise the training set
print(train_df.shape)
print(test_df.shape)
print(train_df['Chromophore'].nunique())
print(test_df['Chromophore'].nunique())

3839
(5970, 4)
(2062, 4)
2839
1000


In [11]:
# creating CSV files for each of the data subsets for future reference and analysis
train_df.to_csv('d4c_train.csv', index=False)
test_df.to_csv('d4c_test.csv', index=False)

In [12]:
# viewing the training set to ensure it saved correctly - also to compare to following test set to view each's immediate chromophore SMILES entries
pd.read_csv('d4c_train.csv').head()

,Chromophore,Solvent,Absorption max (nm),log(e/mol-1 dm3 cm-1)
0,c1ccc2ccccc2c1,C1CCCCC1,286.0,3.58
1,CCCC[Si](C)(C)c1cccc2ccccc12,C1CCCCC1,294.0,3.74
2,C[Si](C)(C)c1ccc([Si](C)(C)C)c2ccccc12,C1CCCCC1,300.0,3.87
3,C[SiH](C)c1ccc([SiH](C)C)c2ccccc12,C1CCCCC1,300.0,3.86
4,CC(C)(C)[Si](C)(C)c1ccc([Si](C)(C)C(C)(C)C)c2c...,C1CCCCC1,302.0,3.91


In [13]:
# viewing the test set to ensure it saved correctly - also to compare to previous training set to view each's immediate chromophore SMILES entries
pd.read_csv('d4c_test.csv').head()

,Chromophore,Solvent,Absorption max (nm),log(e/mol-1 dm3 cm-1)
0,C[Si](C)(C)c1cccc2ccccc12,C1CCCCC1,294.0,3.73
1,C[SiH](C)c1cccc2ccccc12,C1CCCCC1,294.0,3.75
2,CC(C)(C)[Si](C)(C)c1cccc2ccccc12,C1CCCCC1,295.0,3.78
3,CCCCCCCC[Si](C)(C)c1cccc2ccccc12,C1CCCCC1,294.0,3.76
4,C[Si](C)(C)c1ccc(C#N)c2ccccc12,C1CCCCC1,315.0,3.88


### Comments:

Whole Deep4Chem dataset provided by the authors has 8032 entries, authors discuss a 3.8k molecule dataset so investigated whether that was true using the .unique() method - number of unique molecules is 3839 which is close to what authors say and must be how they treated the data going forward

Train and test subsets created so that the latter contained 1000 molecules (as authors state) and the remaining molecules comprise the former - this splits the data into subsets where there is no overlap in chromophore SMILES entries => reducing potential bias in model as the more unique input column will not be seen during both training and predicting processes